# Assignment 04 — Bangla Handwritten Word Recognition System

**Student:** Md. Jahid Gazi  
**Student ID:** 1000128068

This is the **single main notebook** for the assignment. It prepares the environment, downloads the dataset, inspects it, trains a baseline, logs at least two MLflow runs, evaluates the model, and saves artifacts for the Streamlit/Docker application.

**Dataset source:** [BanglaLekha-Isolated, Mendeley Data](https://data.mendeley.com/datasets/hf6sf8zrkc/2)

> **Important scope check:** BanglaLekha-Isolated is an isolated-character dataset. The notebook checks the downloaded structure rather than assuming it contains complete word images. An isolated-character classifier is not the same as complete handwritten-word recognition. The assignment requires complete-word output, so the word-level stage and evaluation must be validated separately.

## 1. Environment setup

In [ ]:
import os, sys, platform, subprocess, importlib.util, json, random, time, zipfile, shutil
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())

# Install missing runtime packages from this notebook.
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "PIL": "pillow",
    "tensorflow": "tensorflow==2.20.0",
    "mlflow": "mlflow",
    "sklearn": "scikit-learn",
    "requests": "requests",
    "tqdm": "tqdm",
}
missing = [pip_name for module, pip_name in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Dependency check complete.")

## 2. Imports, reproducibility and device

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import mlflow
import requests
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

DEVICE = "GPU" if gpus else "CPU"
print("TensorFlow:", tf.__version__)
print("MLflow:", mlflow.__version__)
print("Device:", DEVICE)

## 3. Project paths and dataset download

The dataset is downloaded from the public Mendeley Data record when possible. If the provider changes its download endpoint or requires an interactive confirmation, the cell prints the dataset page link and stops with a clear message. In that case download the archive from the official page and place it in `data/`; then rerun this section.

Dataset page: https://data.mendeley.com/datasets/hf6sf8zrkc/2

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
EXTRACT_DIR = DATA_DIR / "extracted"
ARTIFACT_DIR = BASE_DIR / "artifacts"
MLFLOW_DIR = BASE_DIR / "mlflow"
for d in [DATA_DIR, RAW_DIR, EXTRACT_DIR, ARTIFACT_DIR, MLFLOW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_ID = "hf6sf8zrkc"
DATASET_VERSION = "2"
DATASET_PAGE = f"https://data.mendeley.com/datasets/{DATASET_ID}/{DATASET_VERSION}"
ARCHIVE = RAW_DIR / "banglalekha_isolated.zip"

print("Project:", BASE_DIR)
print("Dataset page:", DATASET_PAGE)

In [ ]:
def download_mendeley_dataset():
    # Official Mendeley Data public API. The API docs specify dataset metadata,
    # ZIP metadata, and a redirect endpoint for downloading the archive.
    api_base = "https://api.data.mendeley.com"
    headers = {"Accept": "application/json"}
    metadata_url = f"{api_base}/datasets/{DATASET_ID}"
    zip_metadata_url = f"{api_base}/datasets/{DATASET_ID}/zip"

    meta_resp = requests.get(
        metadata_url, params={"version": DATASET_VERSION}, headers=headers, timeout=60
    )
    meta_resp.raise_for_status()
    metadata = meta_resp.json()
    print("Dataset:", metadata.get("name", "BanglaLekha-Isolated"))
    print("Published version:", metadata.get("version", DATASET_VERSION))

    zip_resp = requests.get(
        zip_metadata_url, params={"version": DATASET_VERSION}, headers=headers, timeout=60
    )
    zip_resp.raise_for_status()
    zip_info = zip_resp.json()
    print("Archive status:", zip_info.get("status"))
    print("Archive size (reported):", zip_info.get("size"))

    if zip_info.get("status") not in (None, "COMPLETED", "AVAILABLE", "READY"):
        raise RuntimeError(
            f"Mendeley ZIP archive is not ready (status={zip_info.get('status')}). "
            f"Try again later or download from {DATASET_PAGE}"
        )

    download_endpoint = f"{api_base}/datasets/{DATASET_ID}/zip/file_downloaded"
    download_resp = requests.get(
        download_endpoint, params={"version": DATASET_VERSION},
        headers=headers, stream=True, allow_redirects=True, timeout=120
    )
    download_resp.raise_for_status()
    content_type = download_resp.headers.get("content-type", "").lower()
    first_chunk = next(download_resp.iter_content(chunk_size=1024 * 1024), b"")
    if not first_chunk.startswith(b"PK"):
        raise RuntimeError(
            f"Expected a ZIP archive but received content-type={content_type}. "
            f"Download manually from {DATASET_PAGE} and save as {ARCHIVE}"
        )
    with open(ARCHIVE, "wb") as f:
        f.write(first_chunk)
        for chunk in download_resp.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    return ARCHIVE

if not ARCHIVE.exists() or ARCHIVE.stat().st_size == 0:
    try:
        downloaded = download_mendeley_dataset()
        print("Downloaded archive:", downloaded, "size:", downloaded.stat().st_size)
    except Exception as e:
        print("Automatic download was not completed:", repr(e))
        print("If needed, download from the official page and save as:", ARCHIVE)
else:
    print("Dataset archive already exists:", ARCHIVE)


## 4. Extract and locate the image folders

In [ ]:
if ARCHIVE.exists() and ARCHIVE.stat().st_size > 0:
    with zipfile.ZipFile(ARCHIVE) as zf:
        bad = zf.testzip()
        if bad:
            raise RuntimeError(f"Corrupt ZIP member: {bad}")
        zf.extractall(EXTRACT_DIR)
    print("Extracted dataset to:", EXTRACT_DIR)

valid_ext = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
image_dirs = []
for candidate in [EXTRACT_DIR, DATA_DIR]:
    if candidate.exists():
        image_dirs.extend([p for p in candidate.rglob("*")
                           if p.is_dir() and any(
                               q.is_dir() for q in p.iterdir()
                           ) and any(
                               f.is_file() and f.suffix.lower() in valid_ext
                               for f in p.rglob("*")
                           )])

# Prefer a directory whose immediate children look like class folders.
def score_image_root(folder):
    children = [x for x in folder.iterdir() if x.is_dir()]
    return sum(1 for x in children if any(
        f.is_file() and f.suffix.lower() in valid_ext for f in x.rglob("*")
    ))

image_dirs = sorted(set(image_dirs), key=score_image_root, reverse=True)
IMAGE_DIR = image_dirs[0] if image_dirs else None
print("Candidate image roots:", [(str(x), score_image_root(x)) for x in image_dirs[:5]])
if IMAGE_DIR is None:
    print("Could not automatically locate class folders. Inspect extracted files and set IMAGE_DIR manually.")
else:
    print("Selected IMAGE_DIR:", IMAGE_DIR)

## 5. Dataset structure, image counts and samples

In [ ]:
if IMAGE_DIR is None:
    raise FileNotFoundError("Dataset image root not found. Set IMAGE_DIR to the directory containing class subfolders.")
class_dirs = sorted([p for p in IMAGE_DIR.iterdir() if p.is_dir()], key=lambda p: p.name)
rows, file_paths, folder_labels = [], [], []
for folder in class_dirs:
    files = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in valid_ext]
    rows.append({"folder_label": folder.name, "image_count": len(files)})
    file_paths.extend(map(str, files))
    folder_labels.extend([folder.name] * len(files))
class_summary = pd.DataFrame(rows)
display(class_summary)
print("Classes:", len(class_dirs), "| Images:", len(file_paths))

In [ ]:
counts = class_summary.set_index("folder_label")["image_count"]
plt.figure(figsize=(14, 4))
plt.bar(range(len(counts)), counts.values)
plt.xticks(range(len(counts)), counts.index, rotation=90)
plt.title("Images per dataset class")
plt.xlabel("Folder label"); plt.ylabel("Image count")
plt.tight_layout(); plt.show()

sample_indices = np.random.default_rng(SEED).choice(len(file_paths), min(12, len(file_paths)), replace=False)
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax in axes.ravel(): ax.axis("off")
for ax, i in zip(axes.ravel(), sample_indices):
    im = Image.open(file_paths[i]).convert("L")
    ax.imshow(im, cmap="gray")
    ax.set_title(folder_labels[i])
plt.tight_layout(); plt.show()

## 6. Prepare labels and split the data

In [ ]:
class_names = [p.name for p in class_dirs]
class_to_idx = {name: i for i, name in enumerate(class_names)}
y_all = np.array([class_to_idx[x] for x in folder_labels], dtype=np.int32)
paths_all = np.array(file_paths)

train_paths, temp_paths, y_train, y_temp = train_test_split(
    paths_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)
val_paths, test_paths, y_val, y_test = train_test_split(
    temp_paths, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
IMG_SIZE = (64, 64)

def load_images(paths):
    x = np.empty((len(paths), 64, 64, 1), dtype=np.uint8)
    for i, path in enumerate(tqdm(paths, desc="Loading images")):
        x[i, :, :, 0] = np.asarray(
            Image.open(path).convert("L").resize(IMG_SIZE, Image.Resampling.LANCZOS),
            dtype=np.uint8
        )
    return x

x_train = load_images(train_paths)
x_val = load_images(val_paths)
x_test = load_images(test_paths)
print("Train:", x_train.shape, "Validation:", x_val.shape, "Test:", x_test.shape)

## 7. Build the CNN model

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model(num_classes):
    return keras.Sequential([
        layers.Input(shape=(64, 64, 1)),
        layers.Rescaling(1.0 / 255.0),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.30),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])

model = build_model(len(class_names))
model.summary()

## 8. Configure MLflow

In [ ]:
TRACKING_URI = f"sqlite:///{(MLFLOW_DIR / 'mlflow.db').as_posix()}"
EXPERIMENT_NAME = "Bangla_Handwritten_Recognition"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())

## 9. Train experiment 1

In [ ]:
def run_training(run_name, epochs=8, batch_size=64, learning_rate=0.001):
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(len(class_names))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    class EpochLogger(keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            mlflow.log_metrics({
                "train_loss": float(logs.get("loss", 0)),
                "train_accuracy": float(logs.get("accuracy", 0)),
                "val_loss": float(logs.get("val_loss", 0)),
                "val_accuracy": float(logs.get("val_accuracy", 0))
            }, step=epoch + 1)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "architecture": "3-block CNN",
            "image_size": "64x64",
            "num_classes": len(class_names),
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": learning_rate,
            "train_images": len(x_train),
            "validation_images": len(x_val)
        })
        history = model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=epochs, batch_size=batch_size,
            callbacks=[
                EpochLogger(),
                keras.callbacks.EarlyStopping(
                    monitor="val_accuracy", patience=3, restore_best_weights=True
                )
            ],
            verbose=2
        )
        val_loss, val_acc = model.evaluate(x_val, y_val, verbose=0)
        mlflow.log_metric("final_val_accuracy", float(val_acc))
        return model, history, float(val_acc)

model_1, history_1, val_acc_1 = run_training(
    "experiment_1_baseline", epochs=8, batch_size=64, learning_rate=0.001
)
print("Experiment 1 validation accuracy:", val_acc_1)

## 10. Train experiment 2 (hyperparameter comparison)

In [ ]:
model_2, history_2, val_acc_2 = run_training(
    "experiment_2_lower_learning_rate", epochs=10, batch_size=64, learning_rate=0.0005
)
print("Experiment 2 validation accuracy:", val_acc_2)

## 11. Compare MLflow runs

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["start_time DESC"]
)
display(runs)
print("Recorded runs:", len(runs))

## 12. Select model and evaluate on held-out test set

In [ ]:
best_model = model_1 if val_acc_1 >= val_acc_2 else model_2
test_probs = best_model.predict(x_test, batch_size=64, verbose=0)
test_preds = test_probs.argmax(axis=1)
test_accuracy = float(accuracy_score(y_test, test_preds))
print("Isolated-character test accuracy:", round(test_accuracy, 4))
print(classification_report(y_test, test_preds, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, test_preds, labels=np.arange(len(class_names)))
plt.figure(figsize=(max(8, min(16, len(class_names)*0.2)), max(7, min(14, len(class_names)*0.2))))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion matrix — isolated character classes")
plt.xlabel("Predicted class index"); plt.ylabel("True class index")
plt.colorbar(); plt.tight_layout(); plt.show()

## 13. Save model, labels and audit metadata

In [ ]:
MODEL_PATH = ARTIFACT_DIR / "bangla_character_cnn.keras"
LABELS_PATH = ARTIFACT_DIR / "labels.json"
best_model.save(MODEL_PATH)
labels = {
    "class_names": class_names,
    "class_to_index": class_to_idx,
    "index_to_class": {str(i): name for i, name in enumerate(class_names)},
    "label_note": "Folder labels only. Confirm official Unicode mapping before converting predictions to Bangla characters."
}
LABELS_PATH.write_text(json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "dataset_audit.json").write_text(
    json.dumps({
        "dataset_source": DATASET_PAGE,
        "image_root": str(IMAGE_DIR),
        "num_classes": len(class_names),
        "num_images": len(file_paths),
        "class_counts": class_summary.to_dict(orient="records")
    }, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Saved:", MODEL_PATH)
print("Saved:", LABELS_PATH)

## 14. Reload and verify saved model

In [ ]:
reloaded = keras.models.load_model(MODEL_PATH)
p1 = best_model.predict(x_test[:50], verbose=0)
p2 = reloaded.predict(x_test[:50], verbose=0)
print("Reloaded predictions match:", bool(np.allclose(p1, p2, atol=1e-6)))

## 15. Launch MLflow UI

In [ ]:
print("Run this in a separate WSL terminal from the project root:")
print("source .venv/bin/activate")
print("mlflow ui --backend-store-uri", TRACKING_URI, "--host 0.0.0.0 --port 5000")
print("Then open http://localhost:5000")

## 16. Assignment requirement review — whole-word recognition

**Do not treat the above isolated-character classification as the completed assignment.**

- This notebook inspects the actual downloaded dataset folders and labels.
- If each image is one isolated character, it can train a character classifier only.
- Assignment 04 requires a complete handwritten word image to produce the complete Bangla word.
- A word-level dataset or a validated segmentation/sequence-recognition approach and a word-level test set are still required.
- Report character-level metrics separately from complete-word accuracy.
- Capture the MLflow experiment page showing at least two runs.
- The final Streamlit application must display a complete predicted word, and the Docker image must run that application.